# Actividad 4: Acuerdo Inter-Anotador (Evaluación de Qrels)

Este notebook implementa el protocolo de auditoría manual para validar la calidad de los juicios de relevancia generados de manera automatizada mediante un Modelo de Lenguaje de Gran Escala (LLM).

## Marco Teórico y Métricas

Para evaluar la consistencia entre los juicios de relevancia del LLM (escala de 0 a 5) y los juicios de un anotador humano, calculamos dos métricas principales:

### 1. Acuerdo Exacto (Exact Agreement)
Es la proporción simple de pares consulta-documento donde ambos evaluadores asignaron exactamente el mismo score de relevancia.

$$\text{Exact Agreement} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}[y_i^{\text{LLM}} = y_i^{\text{human}}]$$

### 2. Kappa de Cohen ($\kappa$)
El coeficiente Kappa de Cohen mide el acuerdo inter-anotador para variables nominales u ordinales, corrigiendo la fracción de acuerdo esperada por el puro azar.

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

Donde:
- $p_o$ es el acuerdo observado (equivalente al Acuerdo Exacto).
- $p_e$ es el acuerdo esperado por azar, calculado a partir de las distribuciones marginales de cada anotador:

$$p_e = \sum_{k \in \text{labels}} P(\text{LLM} = k) \cdot P(\text{Human} = k)$$

### Interpretación de Kappa (Landis & Koch, 1977)

| Rango de $\kappa$ | Fuerza del Acuerdo |
|---|---|
| $< 0.00$ | Pobre (menor al azar) |
| $0.00 - 0.20$ | Leve (Slight) |
| $0.21 - 0.40$ | Aceptable (Fair) |
| $0.41 - 0.60$ | Moderado (Moderate) |
| $0.61 - 0.80$ | Sustancial (Substantial) |
| $0.81 - 1.00$ | Casi Perfecto (Almost Perfect) |

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Resolución de rutas compatible con el entorno Jupyter y ejecución desde raíz
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

QRELS_PATH = ROOT_DIR / "data" / "qrels_to_grade.csv"
SAMPLE_PATH = ROOT_DIR / "human_review_sample.csv"

SAMPLE_SIZE = 50
SEED = 42
RELEVANCE_COL = "relevance"          # Columna de la evaluación LLM (0-5)
HUMAN_COL = "human_relevance"        # Columna a completar por el humano (0-5)
LABELS = list(range(6))              # [0, 1, 2, 3, 4, 5]

print(f"Ruta del archivo Qrels: {QRELS_PATH.resolve()}")
print(f"Ruta de la muestra humana: {SAMPLE_PATH.resolve()}")

## Paso 1: Generación de la Muestra Estratificada

Para la auditoría manual, extraemos una muestra aleatoria estratificada de 50 pares de consultas e ítems a partir de `qrels_to_grade.csv`.
La **estratificación** nos garantiza que todas las categorías de relevancia (de 0 a 5) estén representadas en la muestra de auditoría, evitando que categorías raras queden fuera por azar.

In [ ]:
def extract_sample(qrels_path=QRELS_PATH, sample_path=SAMPLE_PATH, n=SAMPLE_SIZE, seed=SEED):
    """Extrae una muestra aleatoria estratificada de n filas con columna human_relevance vacía."""
    if not qrels_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo de qrels en {qrels_path}. ¿Ejecutaste los notebooks anteriores?")
        
    df = pd.read_csv(qrels_path)
    print(f"Qrels cargados: {len(df):,} filas")
    print(f"Distribución original de relevancia LLM:\n{df[RELEVANCE_COL].value_counts().sort_index()}\n")

    # Garantizar al menos 1 fila de cada nivel de relevancia existente en el dataset
    rng = np.random.RandomState(seed)
    groups = df.groupby(RELEVANCE_COL)
    n_levels = len(groups)

    if n < n_levels:
        raise ValueError(f"El tamaño de muestra ({n}) es menor al número de clases ({n_levels}).")

    # 1 elemento obligatorio por clase
    mandatory = groups.apply(lambda g: g.sample(1, random_state=rng))
    mandatory = mandatory.droplevel(0)

    # Completar el cupo restante de forma proporcional a la distribución
    remaining_pool = df.drop(mandatory.index)
    remaining_n = n - len(mandatory)

    if remaining_n > 0:
        extra = remaining_pool.sample(n=min(remaining_n, len(remaining_pool)), random_state=rng)
        sample = pd.concat([mandatory, extra]).sample(frac=1, random_state=rng)
    else:
        sample = mandatory

    # Añadir columna vacía para la revisión humana y guardar
    sample = sample.reset_index(drop=True)
    sample[HUMAN_COL] = np.nan

    sample.to_csv(sample_path, index=False)
    print(f"[OK] Muestra estratificada de {len(sample)} filas guardada en: {sample_path.name}")
    print(f"Distribución en la muestra generada:\n{sample[RELEVANCE_COL].value_counts().sort_index()}\n")
    print(f"--> Instrucción: Abre '{sample_path.name}' y completa la columna '{HUMAN_COL}' con valores del 0 al 5.")
    return sample

# Generamos la muestra solo si no existe ya para no pisar el trabajo de anotación previo
if not SAMPLE_PATH.exists():
    extract_sample()
else:
    print(f"El archivo '{SAMPLE_PATH.name}' ya existe en el directorio. Cargando muestra existente...")
    sample_df = pd.read_csv(SAMPLE_PATH)
    missing = sample_df[HUMAN_COL].isna().sum()
    print(f"Muestra cargada: {len(sample_df)} filas. Faltan anotar {missing} filas.")

## Paso 2: Cálculo de Acuerdo Inter-Anotador

Una vez que hayas completado las anotaciones humanas en `human_review_sample.csv`, ejecuta la siguiente celda para procesar las métricas de coincidencia.

In [ ]:
def exact_agreement(llm, human):
    """Calcula la proporción de acuerdos exactos."""
    return float(np.mean(llm == human))

def cohens_kappa(llm, human, labels=LABELS):
    """Calcula el coeficiente Kappa de Cohen de forma manual/vectorizada."""
    n = len(llm)
    if n == 0:
        raise ValueError("Muestra vacía.")
    
    p_o = np.mean(llm == human)
    p_e = 0.0
    for k in labels:
        p_llm_k = np.sum(llm == k) / n
        p_human_k = np.sum(human == k) / n
        p_e += p_llm_k * p_human_k
        
    if p_e == 1.0:
        return 1.0
        
    kappa = (p_o - p_e) / (1.0 - p_e)
    return float(kappa)

def interpret_kappa(kappa):
    """Retorna la interpretación cualitativa según Landis & Koch."""
    if kappa < 0.0:
        return "Pobre (menor al azar)"
    elif kappa <= 0.20:
        return "Leve (Slight)"
    elif kappa <= 0.40:
        return "Aceptable (Fair)"
    elif kappa <= 0.60:
        return "Moderado (Moderate)"
    elif kappa <= 0.80:
        return "Sustancial (Substantial)"
    else:
        return "Casi Perfecto (Almost Perfect)"

def evaluate_agreement(sample_path=SAMPLE_PATH):
    """Carga los datos anotados y calcula el reporte completo."""
    df = pd.read_csv(sample_path)
    
    if HUMAN_COL not in df.columns:
        print(f"ERROR: La columna '{HUMAN_COL}' no existe en el archivo.")
        return
        
    missing = df[HUMAN_COL].isna().sum()
    if missing > 0:
        print(f"ADVERTENCIA: Quedan {missing} filas sin anotar. Evaluando solo las filas completas...\n")
        df = df.dropna(subset=[HUMAN_COL])
        
    if len(df) == 0:
        print("ERROR: No hay filas anotadas para calcular métricas.")
        return
        
    llm = df[RELEVANCE_COL].values.astype(int)
    human = df[HUMAN_COL].values.astype(int)
    
    ea = exact_agreement(llm, human)
    kappa = cohens_kappa(llm, human)
    
    print("=" * 65)
    print("               REPORTE DE ACUERDO INTER-ANOTADOR")
    print("                 LLM (Relevance) vs. Humano")
    print("=" * 65)
    print(f"  Cantidad de pares anotados: {len(df)}")
    print(f"  Acuerdo Exacto (Exact Agreement): {ea:.2%}")
    print(f"  Kappa de Cohen (\kappa): {kappa:.4f} ({interpret_kappa(kappa)})")
    print("=" * 65)
    
    # Matriz de Confusión
    print("\nMatriz de Confusión (Filas: LLM, Columnas: Humano):\n")
    confusion = pd.crosstab(
        pd.Categorical(llm, categories=LABELS),
        pd.Categorical(human, categories=LABELS),
        rownames=['LLM'],
        colnames=['Humano'],
        dropna=False
    )
    print(confusion)
    
    # Análisis de desacuerdos
    disagreements = df[llm != human].copy()
    disagreements['diff'] = (disagreements[HUMAN_COL] - disagreements[RELEVANCE_COL]).astype(int)
    if len(disagreements) > 0:
        print(f"\nDesacuerdos registrados: {len(disagreements)} / {len(df)}")
        print(f"  Diferencia absoluta promedio: {disagreements['diff'].abs().mean():.2f}")
        print(f"  Casos donde el LLM sobrevalora (LLM > Humano): {(disagreements['diff'] < 0).sum()}")
        print(f"  Casos donde el LLM infravalora (LLM < Humano): {(disagreements['diff'] > 0).sum()}")
    else:
        print("\n¡Acuerdo perfecto! No se registraron diferencias de criterio.")

evaluate_agreement()